# Swing ZigZag (ATR-prominence)

__Algorithm:__ ATR-prominence ZigZag swing detector → reversal trade at every confirmed swing → flip on the next opposite-side confirmation. Same parameters work across symbols and regimes because the prominence threshold is in ATR units, not points or percent.

__Pattern:__
1. Track the running max (high) and min (low) since the last confirmed pivot.
2. A candidate confirms once the retrace from the candidate, measured at the candidate's bar, exceeds `min_prominence_atr × ATR(pivot)` — *and* the confirmation cooldown of min_bars_between bars has elapsed since the last confirmation.
3. Confirmed swings strictly alternate `high → low → high → low`.

__Features:__
- Volatility-normalized threshold (works without per-symbol tuning).
- Graded swings carry continuous prominence_atr, volume_z, range_z, plus a composite score; the strategy can gate entries on a minimum score.
- Look-ahead free: at bar i the strategy only consumes swings with `confirmation_idx ≤ i`.

__How the strategy maps swings to trades:__
- Confirmed swing **HIGH** → SHORT entry at that bar's close (top is in; fade the bounce).
- Confirmed swing **LOW**  → LONG  entry at that bar's close (bottom is in; fade the dip).
- Exit on the next opposite-side swing confirmation — which is also the next entry signal, so the strategy flips on each confirmation.
- Optional ATR trailing stop (`swing_zz_stop_atr_mult × ATR`) as a defensive exit when the next swing takes too long to confirm.

## Table of Contents

1. [Configuration](#configuration)
2. [Synthetic correctness check](#synthetic-correctness-check)
3. [Theoretical ceilings](#theoretical-ceilings)
4. [Single-tier detection](#single-tier-detection)
5. [Tiered detection](#tiered-detection)
6. [Inspect](#inspect)
7. [Strategy backtest](#strategy-backtest)
8. [Visualize](#visualize)
9. [Prominence sweep](#prominence-sweep)
10. [Live signals](#live-signals)

## Configuration

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.strategies import SwingZigZagStrategy
from engine.swings import detect_swings, detect_swings_tiered
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Synthetic correctness check

Sanity test on a wave-shaped synthetic series where the swings are obvious. Two invariants any correct run must satisfy:
- **Alternation** — confirmed swings strictly alternate high/low.
- **Non-degenerate output** — many swings on a multi-cycle oscillator.

If this cell errors, the detector is broken — don't trust later results.

In [6]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n_syn = 400
t = np.linspace(0, 8 * np.pi, n_syn)
mid = 100.0 + 20.0 * np.sin(t)
close_syn = mid + rng.standard_normal(n_syn) * 0.3
high_syn = close_syn + np.abs(rng.standard_normal(n_syn)) * 0.6
low_syn  = close_syn - np.abs(rng.standard_normal(n_syn)) * 0.6
vol_syn  = rng.uniform(100, 1000, n_syn)
df_syn = pd.DataFrame(
    {"open": close_syn, "high": high_syn, "low": low_syn,
     "close": close_syn, "volume": vol_syn},
    index=pd.date_range("2026-01-01", periods=n_syn, freq="15min", tz="UTC"),
)

syn_swings = detect_swings(df_syn, min_prominence_atr=1.0, return_provisional=False)
sides = [s.side for s in syn_swings]
assert all(a != b for a, b in zip(sides, sides[1:])), "swings must alternate"
assert len(syn_swings) >= 8, f"sine wave should produce many swings, got {len(syn_swings)}"
print(f"OK — {len(syn_swings)} alternating swings on the synthetic wave")

OK — 77 alternating swings on the synthetic wave


## Theoretical ceilings

Two upper bounds for what this strategy could return on df:

- **Oracle-pivot ceiling** — the strategy's structural maximum; the maximum P&L achievable if the detector could see the future and label every true extremum perfectly. \
Strips causality from the detector: instead of confirming swings forward in time, look at the whole series and pick the true alternating extrema in close. Trade the strategy's existing rules on those pivots and sum the segments. \
This is the most P&L any flip-on-pivot strategy can extract, the structural ceiling for *any* swing-style strategy on this data, regardless of how clever the detector is.
- **Detector-constrained ceiling** — the best in-sample result over a grid search of the detector's parameters. Tighter than the oracle (no causal algorithm can label extrema perfectly), and a more honest target. \
This is what you actually compete with — it tells you whether the room to grow is in parameter tuning (gap small) or in changing the detector itself (gap large).

Both are in-sample maxima — interpret them as the room the live strategy has to grow, not as forecasts. Use `result.total_pnl_bps / oracle_max_bps` as a *skill ratio*: 30–50% is excellent, single digits means the detector is leaving most of the alpha on the table.

### Oracle-pivot ceiling

The vectorized O(n²) DP over close prices with parent-pointer reconstruction; reports pivot count, sum-of-bps, and compounded $100 balance.

DP over close prices: `dp[i]` = max cumulative bps ending with a pivot at bar i; segment P&L is `|Δclose| / close * 10_000 - cost_bps`. The optimum chooses any subset of bars to pivot at, alternating direction implicitly because segment P&L is the absolute change. O(n²) but vectorized — fast for n in the thousands.

In [ ]:
import numpy as np

close = df["close"].to_numpy()
cost_bps = ACTIVE_TRADE.total_cost_bps()
n = len(close)

dp = np.zeros(n)
prev = np.full(n, -1, dtype=int)
for i in range(1, n):
    seg = np.abs(close[i] - close[:i]) / close[:i] * 10_000 - cost_bps
    candidates = dp[:i] + seg
    best_j = int(candidates.argmax())
    best_val = float(candidates[best_j])
    if best_val > 0:
        dp[i] = best_val
        prev[i] = best_j

oracle_max_bps = float(dp.max())

# Reconstruct optimal pivot chain by walking the parent pointers.
chain = []
cur = int(dp.argmax())
while cur != -1:
    chain.append(cur)
    cur = int(prev[cur])
chain = list(reversed(chain))

# Compound $100 through the oracle's segments for an apples-to-apples dollar comparison.
INITIAL_BALANCE = 100.0
balance = INITIAL_BALANCE
for a, b in zip(chain[:-1], chain[1:]):
    seg_bps = abs(close[b] - close[a]) / close[a] * 10_000 - cost_bps
    balance *= (1 + seg_bps / 10_000)

print(f"Round-trip cost      : {cost_bps:.1f} bps")
print(f"Oracle pivots        : {len(chain)} (flips: {len(chain) - 1})")
print(f"Oracle sum-of-bps    : {oracle_max_bps:+,.1f}")

Round-trip cost      : 12.0 bps
Oracle pivots        : 3372 (flips: 3371)
Oracle sum-of-bps    : +147,526.7
Oracle $100 → $236,397,551.71  (+236397451.71%)


__Reading the result:__
- The sum-of-bps number is the honest ceiling: "total bps of price movement you could capture if you saw every wiggle." 
- Compare your strategy's total_pnl_bps against that — if your strategy makes 5,000 bps, your skill ratio is ~3 %, which is the real story.

### Detector-constrained ceiling

84-config grid search over min_prominence_atr × min_bars_between × atr_period; sorts by total P&L, prints the oracle/detector ratio, and shows the top-10 row table.

Grid-search the detector's three structural parameters and report the best in-sample run. Compare with the oracle above to see how much of the ceiling a causal algorithm can recover. Heads-up: this is in-sample optimization — the best row is overfit to df and won't reproduce out-of-sample without shrinkage. For a defensible number, split the data or bootstrap and report a distribution.

In [8]:
import pandas as pd
from dataclasses import replace

PROMINENCE_GRID    = (0.5, 0.8, 1.0, 1.5, 2.0, 2.5, 3.0)
BARS_BETWEEN_GRID  = (1, 2, 3, 5)
ATR_PERIOD_GRID    = (7, 14, 21)

base_config = StrategyConfig()
rows = []
for prom in PROMINENCE_GRID:
    for bb in BARS_BETWEEN_GRID:
        for ap in ATR_PERIOD_GRID:
            cfg = replace(
                base_config,
                swing_zz_min_prominence_atr=prom,
                swing_zz_min_bars_between=bb,
                swing_zz_atr_period=ap,
            )
            r = Backtester(SwingZigZagStrategy(cfg), symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)
            rows.append({
                "min_prominence_atr": prom,
                "min_bars_between": bb,
                "atr_period": ap,
                "trades": r.total_trades,
                "win_rate": round(r.win_rate * 100, 1),
                "total_pnl_bps": round(r.total_pnl_bps, 1),
                "profit_factor": round(r.profit_factor, 2),
                "max_dd_bps": round(r.max_drawdown_bps, 1),
            })

detector_ceiling = (
    pd.DataFrame(rows)
      .sort_values("total_pnl_bps", ascending=False)
      .reset_index(drop=True)
)
detector_max_bps = float(detector_ceiling["total_pnl_bps"].iloc[0])
skill_ratio = detector_max_bps / oracle_max_bps if oracle_max_bps > 0 else float("nan")

print(f"Configs tested        : {len(detector_ceiling)}")
print(f"Oracle ceiling (bps)  : {oracle_max_bps:+,.1f}")
print(f"Detector best  (bps)  : {detector_max_bps:+,.1f}")
print(f"Detector / oracle     : {skill_ratio:.1%}")
detector_ceiling.head(10)

Configs tested        : 84
Oracle ceiling (bps)  : +147,526.7
Detector best  (bps)  : -5,688.9
Detector / oracle     : -3.9%


,min_prominence_atr,min_bars_between,atr_period,trades,win_rate,total_pnl_bps,profit_factor,max_dd_bps
0,3.0,5,7,404,31.9,-5688.9,0.71,6003.2
1,3.0,3,7,420,31.2,-5774.0,0.72,6094.5
2,3.0,1,7,428,30.1,-6583.7,0.69,6901.5
3,3.0,5,14,499,32.7,-6624.2,0.73,7539.7
4,3.0,2,7,426,30.0,-6703.5,0.68,7021.4
5,3.0,5,21,544,33.3,-7765.1,0.71,8013.8
6,3.0,3,14,535,30.7,-8560.8,0.67,8937.8
7,3.0,3,21,588,31.3,-9327.6,0.67,9576.4
8,3.0,2,14,555,30.1,-9363.1,0.66,9749.6
9,3.0,1,14,565,31.2,-9485.5,0.66,9869.3


## Single-tier detection

One detector run at one prominence threshold. The output is a list of Swing records; each carries the pivot bar (idx), the confirmation bar (confirmation_idx), the prominence in ATR units, plus volume/range context and a composite quality score.

In [ ]:
MIN_PROMINENCE_ATR = 1.5
ATR_PERIOD         = 14
MIN_BARS_BETWEEN   = 3

single = detect_swings(
    df,
    atr_period=ATR_PERIOD,
    min_prominence_atr=MIN_PROMINENCE_ATR,
    min_bars_between=MIN_BARS_BETWEEN,
    return_provisional=False,
)

print(f"Swings found: {len(single)}")
if single:
    proms = [s.prominence_atr for s in single]
    lags  = [s.bars_to_confirm for s in single]
    print(f"  avg prominence : {sum(proms)/len(proms):.2f} ATR")
    print(f"  avg bars-to-confirm: {sum(lags)/len(lags):.1f}")

## Tiered detection

The same detector run at three sensitivities — small / medium / large swings. Each tier is an independent run, so a small swing in tier 0.8 may be invisible in tier 2.5. Use this to pick the prominence band that matches the timescale of the trades you want.

In [ ]:
TIERS = (0.8, 1.5, 2.5)

tiered = detect_swings_tiered(
    df,
    tiers=TIERS,
    atr_period=ATR_PERIOD,
    min_bars_between=MIN_BARS_BETWEEN,
    return_provisional=False,
)

for thr, lst in tiered.items():
    print(f"  tier {thr:>4}: {len(lst):3d} swings")

## Inspect

Every swing as a sortable row. Useful for asking *what were the top-N strongest swings this month* — sort by score descending.

In [ ]:
from dataclasses import asdict

rows = []
for thr, lst in tiered.items():
    for s in lst:
        d = asdict(s)
        d["tier"] = thr
        d["timestamp"] = df.index[s.idx]
        rows.append(d)

swings_df = (
    pd.DataFrame(rows)
      .sort_values(["tier", "idx"])
      .reset_index(drop=True)
)
swings_df[[
    "tier", "timestamp", "side", "price", "prominence_atr",
    "bars_to_confirm", "volume_z", "range_z", "score",
]].head(20)

## Strategy backtest

Trade each confirmed swing: SHORT on swing highs, LONG on swing lows, flip on the next opposite-side confirmation. Defaults match the single-tier detection above (1.5σ prominence).

In [ ]:
config = StrategyConfig(
    swing_zz_atr_period=ATR_PERIOD,
    swing_zz_min_prominence_atr=MIN_PROMINENCE_ATR,
    swing_zz_min_bars_between=MIN_BARS_BETWEEN,
)
strategy = SwingZigZagStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

## Visualize

Candlestick chart with the confirmed swing pivots overlaid (▲ green = swing low, ▼ red = swing high) connected by a dotted ZigZag, plus entry/exit triangles from the backtest. Hover a pivot to see its prominence and score.

In [ ]:
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Prominence sweep

Re-run the strategy at a ladder of min_prominence_atr values. Higher = fewer, larger swings → fewer trades but typically higher per-trade conviction. Sorted by total P&L (bps) descending so the most profitable setting is on top.

In [ ]:
from dataclasses import replace

INITIAL_BALANCE = 100  # USD
PROMINENCE_TIERS = (0.8, 1.2, 1.5, 2.0, 2.5, 3.0)

sweep_rows = []
for prom in PROMINENCE_TIERS:
    cfg_k = replace(config, swing_zz_min_prominence_atr=prom)
    strat_k = SwingZigZagStrategy(cfg_k)
    r = Backtester(strat_k, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)

    balance, peak, max_dd = INITIAL_BALANCE, INITIAL_BALANCE, 0.0
    for t in r.trades:
        balance *= (1 + t.pnl_bps / 10_000)
        peak = max(peak, balance)
        max_dd = max(max_dd, (peak - balance) / peak)

    sweep_rows.append({
        "min_prominence_atr": prom,
        "trades": r.total_trades,
        "win_rate": r.win_rate,
        "total_pnl_bps": r.total_pnl_bps,
        "avg_pnl_bps": r.avg_pnl_bps,
        "profit_factor": r.profit_factor,
        "max_dd_bps": r.max_drawdown_bps,
        "final_balance": balance,
        "return_pct": (balance / INITIAL_BALANCE - 1) * 100,
    })

sweep = (
    pd.DataFrame(sweep_rows)
      .sort_values("total_pnl_bps", ascending=False)
      .reset_index(drop=True)
)
sweep["win_rate"] = (sweep["win_rate"] * 100).round(1)
sweep["total_pnl_bps"] = sweep["total_pnl_bps"].round(1)
sweep["avg_pnl_bps"] = sweep["avg_pnl_bps"].round(1)
sweep["profit_factor"] = sweep["profit_factor"].round(2)
sweep["max_dd_bps"] = sweep["max_dd_bps"].round(1)
sweep["final_balance"] = sweep["final_balance"].round(2)
sweep["return_pct"] = sweep["return_pct"].round(2)
sweep

## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart under data/live/ each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy swing_zigzag \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import StrategyConfig
from engine.strategies import SwingZigZagStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = StrategyConfig()
strategy = SwingZigZagStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{strategy.name}.db"),
)

engine.run()  # blocks until Ctrl+C or kernel interrupt